# Nauka o Danych I — Laboratorium 3
## Wizualizacja danych: Matplotlib, Plotly, plotnine oraz mapy

**Wariant 14:** *United States Health-Care Spending Attributable to Modifiable Risk Factors 2016*  
**Dane:** `IHME_USA_RISK_SPENDING_2016_Y2020M09D29.csv`  
**Opracowanie:** na podstawie instrukcji `NaukaoDanychI_4.pdf` (w materiale nazwanej „Zajęcie 4”) oraz przykładowego notebooka `Zajecie4.ipynb`.

Celem jest praktyczne poznanie typów wykresów wymienionych w instrukcji. Większość wykresów wykorzystuje rzeczywiste rekordy wariantu 14. **Przykłady geoprzestrzenne mają osobne, jawnie oznaczone dane demonstracyjne**, ponieważ źródłowy CSV zawiera tylko poziom USA, bez podziału na stany i bez współrzędnych.

> Interpretacja: `mean`, `lower`, `upper` to oszacowanie i granice przedziału dla rekordu. Suma/średnia rekordów nie jest automatycznie całkowitym wydatkiem USA (rekordy mogą obejmować nakładające się przekroje).

## 0. Przygotowanie środowiska

W pierwszej komórce Jupyter, **tylko jeśli brakuje bibliotek**, można uruchomić:

```python
%pip install pandas numpy matplotlib plotly plotnine
```

Następnie **Kernel → Restart Kernel**. Interaktywne wykresy Plotly są wyświetlane w Jupyterze, a eksport do PDF może nie zachować ich interaktywności.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotnine import (ggplot, aes, geom_point, geom_line, geom_col, geom_histogram,
                      geom_smooth, geom_boxplot, facet_wrap, theme_minimal,
                      labs, ggtitle, theme, element_text)
from IPython.display import display
from pathlib import Path

# Rysowanie wykresów Matplotlib bezpośrednio w zeszycie.
%matplotlib inline
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 95})

### Wczytanie i kontrola danych
Dalsze komórki korzystają z jednej ramki `df` i przygotowanych agregatów.

In [ ]:
plik = Path('IHME_USA_RISK_SPENDING_2016_Y2020M09D29.csv')
if not plik.exists():
    raise FileNotFoundError(f'Umieść plik {plik.name} w tym samym folderze co notebook.')
df = pd.read_csv(plik)
for kol in ['mean', 'lower', 'upper']:
    df[kol] = pd.to_numeric(df[kol], errors='coerce')
print('Liczba wierszy / kolumn:', df.shape)
print('Brakujące wartości mean:', df['mean'].isna().sum())
print('Lokalizacje:', df['location_name'].unique().tolist())
display(df.head())

In [ ]:
# Agregujemy na potrzeby wizualizacji — to średnie wartości REKORDÓW, nie sumy wydatków kraju.
ryzyka = (df.groupby('risk_name', as_index=False)['mean'].mean()
            .sort_values('mean', ascending=False))
plec = df.groupby('sex', as_index=False)['mean'].mean()
wiek = df.groupby('age_group_name', as_index=False)['mean'].median()
przyczyny = (df.groupby('cause_name', as_index=False)['mean'].mean()
              .nlargest(10, 'mean').sort_values('mean'))
# Numer porządkowy nie jest datą; brak szeregu czasowego (w CSV jest wyłącznie rok 2016).
print('Lata w danych:', df['year'].unique())
display(ryzyka.head(8))

# 1. Biblioteka Matplotlib
## 1.1 Wykres liniowy
Łączymy średnie rekordów dla kategorii ryzyka **uporządkowanych wartościami**. Oś X oznacza pozycję w rankingu, a nie czas.

In [ ]:
top = ryzyka.head(10).reset_index(drop=True)
fig, ax = plt.subplots()
ax.plot(np.arange(1, len(top)+1), top['mean'], marker='o', linewidth=2)
ax.set(xlabel='Pozycja w rankingu ryzyk', ylabel='Średnia wartość rekordu (USD)',
       title='Matplotlib — profil 10 kategorii ryzyka')
ax.set_xticks(np.arange(1, len(top)+1))
plt.tight_layout(); plt.show()

## 1.2 Wykres słupkowy
Porównanie średniej wartości `mean` w wybranych kategoriach ryzyka.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top['risk_name'][::-1], top['mean'][::-1], color='steelblue')
ax.set(xlabel='Średnia wartość rekordu (USD)', title='Matplotlib — TOP 10 kategorii ryzyka')
plt.tight_layout(); plt.show()

## 1.3 Histogram
Histogram przedstawia rozkład `mean` w poszczególnych rekordach.

In [ ]:
fig, ax = plt.subplots()
ax.hist(df['mean'].dropna(), bins=35, color='slateblue', alpha=.75)
ax.axvline(df['mean'].median(), color='orangered', linestyle='--', label='Mediana')
ax.set(xlabel='Wartość rekordu (USD)', ylabel='Liczba rekordów', title='Matplotlib — histogram mean')
ax.legend(); plt.tight_layout(); plt.show()

## 1.4 Wykres kołowy
Udziały **liczby rekordów** w podziale na płeć (a nie udział w wydatkach).

In [ ]:
licznosc = df['sex'].value_counts()
fig, ax = plt.subplots()
ax.pie(licznosc.values, labels=licznosc.index, autopct='%1.1f%%', startangle=90)
ax.set_title('Matplotlib — udział rekordów według płci')
plt.show()

## 1.5 Kolory, style linii i markerów

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(1, 11), top['mean'], color='forestgreen', marker='x',
        linestyle='--', linewidth=2, markersize=8)
ax.set(xlabel='Pozycja w rankingu', ylabel='Średnia mean (USD)', title='Matplotlib — kolor, linia, marker')
plt.tight_layout(); plt.show()

## 1.6 Dodawanie siatki

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(1, 11), top['mean'], marker='o')
ax.grid(True, linestyle=':', alpha=.6)
ax.set(xlabel='Pozycja w rankingu', ylabel='Średnia mean (USD)', title='Matplotlib — wykres z siatką')
plt.tight_layout(); plt.show()

## 1.7 Dodawanie adnotacji

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(1, 11), top['mean'], marker='o')
ax.annotate('Najwyższa średnia', xy=(1, top.iloc[0]['mean']),
            xytext=(2.5, top.iloc[0]['mean']*.87),
            arrowprops={'arrowstyle':'->', 'color':'black'})
ax.set(xlabel='Pozycja w rankingu', ylabel='Średnia mean (USD)', title='Matplotlib — adnotacja')
plt.tight_layout(); plt.show()

## 1.8 Wykres punktowy (scatter)
`mean` i `upper` opisują ten sam szacowany rekord, więc silna zależność jest oczekiwana.

In [ ]:
fig, ax = plt.subplots()
sc=ax.scatter(df['mean'], df['upper'], s=20, c=df['lower'], cmap='viridis', alpha=.55)
fig.colorbar(sc, ax=ax, label='Dolna granica (USD)')
ax.set(xlabel='mean (USD)', ylabel='upper (USD)', title='Matplotlib — zależność mean vs upper')
plt.tight_layout(); plt.show()

## 1.9 Wykres powierzchniowy 3D
Przykład matematyczny zgodny z instrukcją (nie jest to wizualizacja danych CSV).

In [ ]:
x3 = np.linspace(-5, 5, 65)
y3 = np.linspace(-5, 5, 65)
X, Y = np.meshgrid(x3, y3)
Z = np.sin(np.sqrt(X**2 + Y**2))
fig=plt.figure(figsize=(8,5))
ax=fig.add_subplot(111, projection='3d')
ax.plot_surface(X,Y,Z,cmap='plasma',linewidth=0)
ax.set(xlabel='X', ylabel='Y', zlabel='Z', title='Matplotlib — powierzchnia 3D (przykład)')
plt.tight_layout(); plt.show()

# 2. Biblioteka Plotly
Wykresy w tej sekcji są interaktywne w Jupyter. Bazujemy na tych samych rzeczywistych danych z CSV.

## 2.1 Wykres liniowy

In [ ]:
fig=px.line(top.assign(pozycja=range(1,11)), x='pozycja', y='mean', markers=True,
            hover_name='risk_name', title='Plotly — średnie według rankingu ryzyka',
            labels={'pozycja':'Pozycja w rankingu','mean':'Średnia rekordu (USD)'})
fig.show()

## 2.2 Wykres słupkowy

In [ ]:
fig=px.bar(top, x='mean', y='risk_name', orientation='h',
           title='Plotly — 10 kategorii ryzyka', labels={'risk_name':'Ryzyko','mean':'Średnia rekordu (USD)'})
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

## 2.3 Wykres kołowy

In [ ]:
fig=px.pie(names=licznosc.index, values=licznosc.values,
           title='Plotly — udział liczby rekordów według płci')
fig.show()

## 2.4 Histogram

In [ ]:
fig=px.histogram(df, x='mean', nbins=35,
                 title='Plotly — rozkład mean', labels={'mean':'Wartość rekordu (USD)'})
fig.show()

## 2.5 Kolory i styl

In [ ]:
fig=px.line(top.assign(pozycja=range(1,11)), x='pozycja', y='mean', markers=True,
            title='Plotly — dopasowany styl')
fig.update_traces(line={'color':'purple', 'width':4}, marker={'size':9})
fig.show()

## 2.6 Dodawanie adnotacji

In [ ]:
fig=px.line(top.assign(pozycja=range(1,11)), x='pozycja', y='mean', markers=True,
            title='Plotly — adnotacja maksimum')
fig.add_annotation(x=1, y=float(top.iloc[0]['mean']), text='Najwyższa średnia',
                   showarrow=True, arrowhead=2, ay=-45)
fig.show()

## 2.7 Wykres punktowy

In [ ]:
fig=px.scatter(df, x='mean', y='upper', color='sex',
               hover_data=['risk_name','cause_name'],
               title='Plotly — mean i upper według płci')
fig.show()

## 2.8 Wykres powierzchniowy 3D
Przykład matematyczny (jak w instrukcji), niezależny od CSV.

In [ ]:
fig=go.Figure(go.Surface(x=X, y=Y, z=Z, colorscale='Plasma'))
fig.update_layout(title='Plotly — powierzchnia 3D (przykład)',
                  scene={'xaxis_title':'X','yaxis_title':'Y','zaxis_title':'Z'},
                  height=530)
fig.show()

## 2.9 Subploty (kilka wykresów w jednym oknie)

In [ ]:
fig=make_subplots(rows=1, cols=2, subplot_titles=('Średnia wg płci','Mediana wg grupy wieku'))
fig.add_trace(go.Bar(x=plec['sex'],y=plec['mean'],name='Płeć'),row=1,col=1)
fig.add_trace(go.Bar(x=wiek['age_group_name'],y=wiek['mean'],name='Wiek'),row=1,col=2)
fig.update_layout(title='Plotly — porównanie podgrup',showlegend=False,height=480)
fig.update_xaxes(tickangle=-35,row=1,col=2)
fig.show()

# 3. Biblioteka plotnine (Grammar of Graphics)
W tej sekcji korzystamy ze składni `ggplot + aes + geom_*`, zgodnie z instrukcją.

## 3.1 Podstawy — wykres punktowy

In [ ]:
p = ggplot(df, aes(x='mean', y='upper')) + geom_point(alpha=.4) +     labs(title='plotnine — mean vs upper',x='mean (USD)',y='upper (USD)') + theme_minimal()
display(p)

## 3.2 Wykres liniowy

In [ ]:
top_plot = top.assign(pozycja=np.arange(1,len(top)+1))
p = (ggplot(top_plot, aes('pozycja','mean')) + geom_line(color='blue') + geom_point() +
     labs(title='plotnine — średnia według rankingu ryzyka',x='Pozycja',y='Średnia (USD)'))
display(p)

## 3.3 Wykres słupkowy

In [ ]:
p=(ggplot(top, aes(x='risk_name',y='mean')) + geom_col(fill='skyblue') +
   labs(title='plotnine — TOP 10 ryzyk',x='Czynnik ryzyka',y='Średnia (USD)') +
   theme(axis_text_x=element_text(rotation=70, ha='right')))
display(p)

## 3.4 Histogram

In [ ]:
p=ggplot(df,aes(x='mean')) + geom_histogram(bins=35, fill='purple',alpha=.7) +   labs(title='plotnine — histogram mean',x='mean (USD)',y='Liczba rekordów')
display(p)

## 3.5 Kolory i style

In [ ]:
p=(ggplot(df,aes('mean','upper')) + geom_point(color='red',size=1.5,alpha=.4) +
  labs(title='plotnine — kolor i wielkość punktów'))
display(p)

## 3.6 Dodawanie linii trendu

In [ ]:
p=(ggplot(df,aes('mean','upper')) + geom_point(alpha=.25) +
   geom_smooth(method='lm',se=False,color='darkblue') +
   labs(title='plotnine — linia trendu liniowego'))
display(p)

## 3.7 Facetowanie według płci

In [ ]:
p=(ggplot(df,aes('mean','upper')) + geom_point(alpha=.35,size=.7) +
   facet_wrap('~sex') + labs(title='plotnine — osobne panele według płci'))
display(p)

## 3.8 Wykres pudełkowy (box plot)

In [ ]:
p=(ggplot(df,aes(x='sex',y='mean')) + geom_boxplot(fill='lightblue') +
   labs(title='plotnine — rozkład mean według płci',x='Płeć',y='mean (USD)'))
display(p)

## 3.9 Motywy wykresów

In [ ]:
p=(ggplot(df,aes('mean','upper'))+geom_point(alpha=.3)+
   labs(title='plotnine — motyw minimalistyczny')+theme_minimal())
display(p)

# 4. Wizualizacje geoprzestrzenne w Plotly

**Ograniczenie danych:** plik wariantu 14 ma `location_name = United States of America` we wszystkich 1241 wierszach; **nie ma** kolumn `latitude`, `longitude` ani kodów stanów. Z tego powodu nie wolno przypisywać poszczególnych wydatków do stanów i miast.

W następnych przykładach:
- mapa punktowa korzysta z **rzeczywistych współrzędnych miast demonstracyjnych**, ale nie z wydatków;
- mapa krajów pokazuje **liczbę rekordów dla USA**;
- mapa gęstości i mapa stanów korzystają z **danych syntetycznych/demonstracyjnych**, jawnie oznaczonych i niepowiązanych z CSV.

## 4.1 Mapa punktowa — cztery miasta demonstracyjne

In [ ]:
miasta=pd.DataFrame({'miasto':['Nowy Jork','Londyn','Tokio','Sydney'],
 'lat':[40.7128,51.5074,35.6895,-33.8688],
 'lon':[-74.0060,-0.1278,139.6917,151.2093]})
fig=px.scatter_map(miasta,lat='lat',lon='lon',hover_name='miasto',zoom=0,
                   title='Mapa punktowa — współrzędne miast (przykład)',height=440)
fig.update_layout(map_style='open-street-map')
fig.show()

## 4.2 Mapa choropleth — liczba rekordów z CSV dla USA
To jedyny poziom geograficzny dostępny w zbiorze. Kolor dotyczy **liczby rekordów**, nie kosztów.

In [ ]:
kraje=pd.DataFrame({'iso3':['USA'],'liczba_rekordow':[len(df)]})
fig=px.choropleth(kraje,locations='iso3',color='liczba_rekordow',scope='north america',
                  color_continuous_scale='Viridis',title='CSV — liczba rekordów: USA (jeden kraj)')
fig.show()

## 4.3 Mapa gęstości — syntetyczne punkty
Wygenerowane deterministycznie wokół San Francisco; **nie przedstawiają pacjentów ani rzeczywistych wydatków**.

In [ ]:
rng=np.random.default_rng(14)
probka_geo=pd.DataFrame({'lat':rng.normal(37.7749,.055,170),
                        'lon':rng.normal(-122.4194,.065,170)})
fig=px.density_map(probka_geo,lat='lat',lon='lon',radius=18,zoom=9,
                   title='Mapa gęstości — dane syntetyczne (przykład)',height=470)
fig.update_layout(map_style='open-street-map')
fig.show()

## 4.4 Mapa stanów USA — dane demonstracyjne
W instrukcji przykład danych „stanów” bazuje na krajach; tutaj zastosowano poprawne **kody stanów**, ale wartości są wyłącznie sztuczne.

In [ ]:
stany_demo=pd.DataFrame({'stan':['CA','TX','FL','NY','IL'],
                               'wartosc_demo':[10,7,6,8,5]})
fig=px.choropleth(stany_demo,locations='stan',locationmode='USA-states',
                  color='wartosc_demo',scope='usa',color_continuous_scale='Blues',
                  title='Stany USA — WARTOŚCI DEMONSTRACYJNE (nie z CSV)')
fig.show()

# 5. Dodatkowa eksploracja zbioru wariantu 14
Instrukcja kończy się poleceniem utworzenia **wszystkich możliwych wykresów w celu eksploracji zbioru danych**. Oprócz przykładów trzech bibliotek przedstawiamy dwie dodatkowe przydatne wizualizacje.

## 5.1 Top 10 przyczyn — poziomy średnich

In [ ]:
fig,ax=plt.subplots(figsize=(10,6))
ax.barh(przyczyny['cause_name'], przyczyny['mean'],color='teal')
ax.set(title='TOP 10 przyczyn według średniej wartości rekordu',xlabel='Średnia mean (USD)')
plt.tight_layout();plt.show()

## 5.2 Zależność oszacowania i szerokości przedziału
Szerokość = `upper − lower`; wykres pomaga sprawdzić, jak niepewność zmienia się wraz z wartością rekordu.

In [ ]:
df_analiza=df.assign(szerokosc_przedzialu=df['upper']-df['lower'])
fig,ax=plt.subplots()
ax.scatter(df_analiza['mean'],df_analiza['szerokosc_przedzialu'],alpha=.4,s=15)
ax.set(xlabel='mean (USD)',ylabel='upper − lower (USD)',
       title='Zależność wartości i rozpiętości przedziału')
plt.tight_layout();plt.show()

# 6. Wnioski

1. **Matplotlib** sprawdza się do prostych, statycznych wykresów i precyzyjnej kontroli wyglądu.
2. **Plotly** umożliwia interaktywną analizę punktów i kategorii, w tym subploty, wykresy 3D oraz mapy.
3. **plotnine** pozwala budować wykresy warstwowo w stylu Grammar of Graphics.
4. Rozkład `mean` jest silnie prawostronnie asymetryczny; same średnie grup wymagają ostrożnej interpretacji.
5. Korelacja `mean` z `upper` jest naturalnie duża, ponieważ to elementy **tego samego oszacowania**.
6. Z uwagi na brak współrzędnych i kodów stanów w źródłowym CSV nie można tworzyć map z wydatkami w poszczególnych stanach czy miastach. Pokazane mapy to demonstracja narzędzi, z wyjątkiem mapy liczby rekordów dla USA.

**Uruchomienie:** umieść CSV obok `.ipynb`, zainstaluj biblioteki wskazane w sekcji 0 i wybierz w Jupyter **Run → Run All Cells**.